# Inkling 2 Mini 정리: 루프 트랜스포머가 무엇인지, Inkling 2 Small과 뭐가 다른지, 32GB 맥·llama.cpp·Ollama로 실행하는 법과 Qwen3.8-27B 비교 (Inkling 2 Small 글의 후속)

위에서부터 차례로 실행해 보세요.

### llama.cpp(Unsloth 프리빌트)와 GGUF 받기

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/llama /content/gguf
curl -sL -o /content/llama/app.tar.gz https://github.com/unslothai/llama.cpp/releases/download/b12588-mix-c0c3de8/app-b12588-mix-c0c3de8-linux-x64-cpu.tar.gz
tar xzf /content/llama/app.tar.gz -C /content/llama
curl -L -o /content/gguf/Inkling-2-Mini-UD-IQ1_S.gguf https://huggingface.co/unsloth/Inkling-2-Mini-GGUF/resolve/main/Inkling-2-Mini-UD-IQ1_S.gguf
/content/llama/llama-server --version

### llama-server 띄우고 첫 응답과 속도 확인

In [ ]:
%%bash
cd /content/llama
nohup ./llama-server -m /content/gguf/Inkling-2-Mini-UD-IQ1_S.gguf --jinja -c 4096 -t $(nproc) --port 8080 --host 127.0.0.1 > /content/server.log 2>&1 &
for i in $(seq 1 120); do curl -sf http://127.0.0.1:8080/health >/dev/null && break; sleep 2; done
cat > /content/req.json <<'E'
{"messages":[{"role":"user","content":"Python으로 문자열이 팰린드롬인지 확인하는 함수를 한 개 작성해 줘. 짧게."}],"max_tokens":300}
E
curl -s http://127.0.0.1:8080/v1/chat/completions -H 'Content-Type: application/json' -d @/content/req.json | jq '{content:.choices[0].message.content, usage, tps:.timings.predicted_per_second}'
ps -o rss,cmd -C llama-server | cut -c1-70

### GGUF 헤더에서 루프 구조 읽기

In [ ]:
%%bash
cat > /content/meta.py <<'E'
import sys,struct
f=open(sys.argv[1],'rb')
def rd(fmt):
    n=struct.calcsize(fmt); return struct.unpack('<'+fmt,f.read(n))[0]
def rs(): n=rd('Q'); return f.read(n).decode('utf8','replace')
F={0:'B',1:'b',2:'H',3:'h',4:'I',5:'i',6:'f',7:'?',10:'Q',11:'q',12:'d'}
def val(t):
    if t in F: return rd(F[t])
    if t==8: return rs()
    if t==9:
        et=rd('I'); n=rd('Q'); return [val(et) for _ in range(n)]
assert f.read(4)==b'GGUF'
ver=rd('I'); nt=rd('Q'); nk=rd('Q')
print('tensors',nt,'kv',nk)
for _ in range(nk):
    k=rs(); t=rd('I'); v=val(t)
    if k.startswith('tokenizer.'): continue
    if any(x in k for x in ['block_count','expert','loop','architecture']): print(k,'=',str(v)[:80])
names=[]
for _ in range(nt):
    n=rs(); d=rd('I'); [rd('Q') for _ in range(d)]; rd('I'); rd('Q'); names.append(n)
print('blk 0..',max(int(n.split('.')[1]) for n in names if n.startswith('blk.')))
E
python3 -I /content/meta.py /content/gguf/Inkling-2-Mini-UD-IQ1_S.gguf